<a href="https://colab.research.google.com/github/project-ccap/project-ccap.github.io/blob/master/2026notebooks/2026_0614sbert_ccap.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
try:
    import fugashi
except ImportError:
    !pip install "fugashi[unidic-lite]"


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.4/47.4 MB 14.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 694.9/694.9 kB 24.2 MB/s eta 0:00:00
  Created wheel for unidic-lite: filename=unidic_lite-1.0.8-py3-none-any.whl size=47658817 sha256=11d9d6c6216a98b2eb26989aa9046e36b5c19b993bc8fe2b02c200dabd3640d8
  Stored in directory: /root/.cache/pip/wheels/5e/1f/0f/4d43887e5476d956fae828ee9b6687becd5544d68b51ed633d
Successfully built unidic-lite


In [2]:
import torch
from transformers import AutoTokenizer, BertModel  # <- AutoTokenizer をインポート
from scipy.stats import pearsonr

class SentenceBertJapanese:
    def __init__(self, model_name_or_path, device=None):
        # 💡 ここを BertJapaneseTokenizer から AutoTokenizer に変更
        self.tokenizer = AutoTokenizer.from_pretrained(model_name_or_path)
        self.model = BertModel.from_pretrained(model_name_or_path)
        self.model.eval()

        if device is None:
            device = "cuda" if torch.cuda.is_available() else "cpu"
        self.device = torch.device(device)
        self.model.to(device)

    def _mean_pooling(self, model_output, attention_mask):
        token_embeddings = model_output[0]
        input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
        return torch.sum(token_embeddings * input_mask_expanded, 1) / torch.clamp(input_mask_expanded.sum(1), min=1e-9)

    @torch.no_grad()
    def encode(self, sentences, batch_size=8):
        all_embeddings = []
        iterator = range(0, len(sentences), batch_size)
        for batch_idx in iterator:
            batch = sentences[batch_idx:batch_idx + batch_size]

            # 💡 self.tokenizer.batch_encode_plus(...) を self.tokenizer(...) に変更
            encoded_input = self.tokenizer(
              batch,
              padding="longest",
              truncation=True, return_tensors="pt").to(self.device)

            model_output = self.model(**encoded_input)
            sentence_embeddings = self._mean_pooling(
              model_output,
              encoded_input["attention_mask"]).to('cpu')

            all_embeddings.extend(sentence_embeddings)

        return torch.stack(all_embeddings)

MODEL_NAME = "sonoisa/sentence-bert-base-ja-mean-tokens-v2"
model = SentenceBertJapanese(MODEL_NAME)


config.json:   0%|          | 0.00/667 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/466 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/258k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/442M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [8]:
sentences = ["暴走したAI", "暴走した人工知能"]
sebtebces = ["テクテク歩く", "ザーザー降る"]

sentence_embeddings = model.encode(sentences, batch_size=8)
##print("Sentence embeddings:", sentence_embeddings)

print(f'相関係数:{pearsonr(sentence_embeddings[0], sentence_embeddings[1])[0]:.3f}')

相関係数:0.487


In [20]:
from sklearn.metrics.pairwise import cosine_similarity
import time

cartoon1 =['男の人が歩いている','男の人がテクテク歩いている',
'帽子をかぶった男が歩いている','帽子をかぶった男がスタスタ歩いている',
'杖をついた男が散歩している','コツコツ杖をついた男が散歩している',
'杖をついた帽子の男が歩く','杖をついた帽子の男が歩く',
'男が出かけている','男がフラッと出かけている',
'男が杖をついている','男がカツカツと杖をついている',
'男が外出している','男がフラリと外出している',
'男がどこかに行くところ','男がフラフラどこかに行くところ',
'男性が歩いている','トコトコ歩く',
'帽子をかぶった男性が歩いている','スタスタ歩く',
'杖を持った男性が歩いている','男性がトコトコ歩いている',
'帽子をかぶって杖を持った男性が歩いている','帽子をかぶった男性がトコトコ歩いている',
'歩いている人がいる','帽子をかぶって杖を持った男性がトコトコ歩いている',
'今日は歩いていた','今日はトコトコ歩こう',
'ある日、外に出かけた','ステッキを持ってスタスタ歩こう',
'杖を持って外に出た','外に出てスタスタ歩いている男性',
'外に出かけて歩く','ある日、スタスタ歩いている男性がいた',
'今日は歩いてみよう','タッタッと歩く男性',
'帽子をかぶった男性が歩いている','トコトコ歩いている男の人',
'帽子をかぶった人が歩いている','トコトコ歩いている男性',
'男性が杖をついて歩いている','杖をついてトコトコ歩いている男性',
'男性がステッキをついて歩いている','トットッと歩く男性',
'男が歩いている','スタスタ歩く男性',
'男性が散歩している','トコトコ軽快に歩く男性',
'散歩をしている男性がいる','男の人はトコトコ歩いている',
'主人公は歩いている','スタスタと杖を突いて歩く男性'
]

cartoon1_embeddings = model.encode(cartoon1[:10], batch_size=8)
print(cosine_similarity(cartoon1_embeddings))


[[0.9999999  0.59696627 0.6700709  0.47638035 0.68149096 0.6130574
  0.5788399  0.5788399  0.5629524  0.28860158]
 [0.59696627 1.0000002  0.4443246  0.4095415  0.43839836 0.42922187
  0.3948853  0.3948853  0.45528537 0.23056827]
 [0.6700709  0.4443246  1.0000002  0.71807605 0.546312   0.5419227
  0.83817065 0.83817065 0.45706642 0.27432093]
 [0.47638035 0.4095415  0.71807605 0.9999999  0.43783158 0.45801014
  0.6400798  0.6400798  0.33821255 0.27983302]
 [0.68149096 0.43839836 0.546312   0.43783158 1.         0.9365508
  0.7640518  0.7640518  0.4494088  0.26916564]
 [0.6130574  0.42922187 0.5419227  0.45801014 0.9365508  0.99999976
  0.76632774 0.76632774 0.45278984 0.32105428]
 [0.5788399  0.3948853  0.83817065 0.6400798  0.7640518  0.76632774
  0.9999997  0.9999997  0.3720513  0.20373935]
 [0.5788399  0.3948853  0.83817065 0.6400798  0.7640518  0.76632774
  0.9999997  0.9999997  0.3720513  0.20373935]
 [0.5629524  0.45528537 0.45706642 0.33821255 0.4494088  0.45278984
  0.3720513  0.

In [25]:
slta_sn = [
'男の人が歩いている',
'帽子をかぶった男が歩いている',
'杖をついた男が散歩している',
'杖をついた帽子の男が歩く',
'男が出かけている',
'男が杖をついている',
'男が外出している',
'男がどこかに行くところ',
'男性が歩いている',
'帽子をかぶった男性が歩いている',
'杖を持った男性が歩いている',
'帽子をかぶって杖を持った男性が歩いている',
'歩いている人がいる',
'今日は歩いていた',
'ある日、外に出かけた',
'杖を持って外に出た',
'外に出かけて歩く',
'今日は歩いてみよう',
'帽子をかぶった男性が歩いている',
'帽子をかぶった人が歩いている',
'男性が杖をついて歩いている',
'男性がステッキをついて歩いている',
'男が歩いている',
'男性が散歩している',
'散歩をしている男性がいる',
'主人公は歩いている',
'風が吹いて帽子が飛ばされる',
'風で帽子が吹き飛んだ',
'風で帽子が舞った',
'風で帽子が吹き飛んだ',
'風で帽子が吹き飛ばされた',
'帽子が飛ばされて驚いた',
'風で帽子がどこかへ飛んだ',
'帽子が飛ばされた',
'男性の帽子が飛ばされた',
'男性の帽子が飛ばされて、驚く',
'風が吹いて、男性の帽子が飛ばされた',
'帽子が飛んでいった',
'帽子が飛んでいく',
'風で男性の帽子が飛ばされた',
'男性の帽子が風で飛ばされ、男性は驚いた',
'風が吹いて、帽子が飛ばされたので驚いた',
'風で帽子が宙に浮く',
'風で帽子が飛ばされる',
'突風で帽子が飛ばされ驚く男性',
'風が吹いて帽子が飛ばされ思わぬ表情',
'風で帽子が飛ばされ驚く男性',
'帽子が飛ばされるハプニングにあう',
'帽子が飛ばされ驚く男性',
'帽子が飛ばされた',
'帽子が転がる',
'川に帽子が落ちる',
'帽子が川に落ちた',
'川に落ちた帽子を男が取りに行く',
'帽子を慌てて取りに行っているところ',
'男が帽子を取りに行った',
'飛ばされた帽子をつかみに行った',
'転がった帽子は川の方に行った',
'帽子を追いかけた',
'帽子を追いかけた男性は水辺まで来た',
'急いで帽子を追いかけた',
'帽子は水辺まで飛んでいった',
'転がる帽子は海まで来た',
'帽子を追いかけて、水の傍まで来てしまった',
'男性は急いで帽子を追いかけた',
'男性は転がる帽子を捕まえに走った',
'帽子は海の近くまで飛んでいった',
'飛んだ帽子が、もうすぐ海に落ちてしまう',
'帽子が転がっていく',
'水辺に向かって帽子が飛ばされる',
'飛ばされた帽子は水辺の方へ',
'水辺に飛ばされた帽子を取りに行く男性',
'飛ばされた帽子を追いかける男性',
'帽子は川の方へ飛ばされる',
'急いで帽子を拾いにいく男性',
'転がる帽子は川の方へいった',
'帽子をステッキで拾い上げた',
'帽子を杖ですくった',
'杖で帽子をひっかけて取った',
'川に落ちた帽子を杖で拾っているところ',
'帽子を取り戻した',
'杖で帽子を拾い上げた',
'帽子を杖で取った',
'帽子をステッキで拾い上げた',
'海に落ちた帽子を杖で拾った',
'男性は杖を使って水に落ちた帽子を拾い上げた',
'男性は杖で帽子を持ち上げた',
'帽子は川に落ちてしまい、杖で拾った',
'帽子は海に落ちてしまったが、杖を使って拾うことができた',
'男性は濡れた帽子を拾い上げた',
'杖の曲がったところを使って帽子を持ち上げた',
'杖を持ち替え、帽子を引っかけた',
'杖で帽子を引っかけて、水に落ちた帽子を救った',
'結局、帽子は落ちてしまったが持っていた杖で拾いあげた',
'水辺に落ちた帽子を持っていた杖で拾う男性',
'川に落ちた帽子を杖で拾いあげる男性',
'持っていた杖で帽子を拾い上げる',
'水辺に落ちてしまった帽子を拾い上げる',
'帽子は川に落ちてしまったが何とか杖で拾いあげる',
'帽子は川に落ちたけど、杖で拾った'
]

t0 = time.perf_counter()
slta_sn_embeddings = model.encode(slta_sn, batch_size=8)
#print(cosine_similarity(cartoon1_embeddings))
t1 = time.perf_counter()
print(f'処理時間:{t1-t0:.3f}秒')


処理時間:7.511秒


In [27]:
import numpy as np
from sklearn.decomposition import PCA

X = np.array(slta_sn_embeddings)
pca = PCA(n_components=99)

# データを次元削減
X_reduced = pca.fit_transform(X)

# 結果の確認
# print("変換後のデータ:")
# print(X_reduced)
print("\n寄与率:")
print(pca.explained_variance_ratio_)


寄与率:
[2.8917116e-01 1.3608584e-01 9.2825644e-02 6.9757327e-02 5.1530462e-02
 4.0022396e-02 2.9345257e-02 2.6166173e-02 2.0076109e-02 1.8024664e-02
 1.6020484e-02 1.5472655e-02 1.3700269e-02 1.2982386e-02 1.0742010e-02
 1.0523460e-02 9.9309310e-03 9.2341891e-03 8.0485363e-03 7.4023549e-03
 6.6867913e-03 6.5679322e-03 5.9506739e-03 5.3629465e-03 4.8527005e-03
 4.3128957e-03 4.1679624e-03 3.9556851e-03 3.7727943e-03 3.5473506e-03
 3.4678222e-03 3.2692272e-03 3.0462563e-03 2.9471780e-03 2.6959283e-03
 2.5398908e-03 2.2872698e-03 2.1319923e-03 2.0842450e-03 1.9792686e-03
 1.9294248e-03 1.8288860e-03 1.7375785e-03 1.6813665e-03 1.5961432e-03
 1.5638185e-03 1.4117397e-03 1.3468743e-03 1.3026420e-03 1.2757657e-03
 1.2024215e-03 1.1431230e-03 1.1184273e-03 1.0985258e-03 1.0592181e-03
 9.8302506e-04 8.9104922e-04 8.6964946e-04 8.4653951e-04 7.3645724e-04
 7.2759541e-04 6.8869948e-04 6.8084977e-04 6.3780061e-04 5.9292803e-04
 5.6401483e-04 5.5672019e-04 5.3975650e-04 4.9687858e-04 4.6713676e-04


In [28]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# cartoon1 全50文(25ペア×2)をエンコード
cartoon1_embeddings_all = model.encode(cartoon1, batch_size=8)

# 通常文(偶数index) とオノマトペ文(奇数index)のペアごとにcos類似度を計算
pair_similarities = []
pair_texts = []

for i in range(0, len(cartoon1), 2):
    normal_text = cartoon1[i]
    onomatope_text = cartoon1[i + 1]

    v_normal = cartoon1_embeddings_all[i].reshape(1, -1)
    v_onomatope = cartoon1_embeddings_all[i + 1].reshape(1, -1)

    sim = cosine_similarity(v_normal, v_onomatope)[0, 0]

    pair_similarities.append(sim)
    pair_texts.append((normal_text, onomatope_text))

pair_similarities = np.array(pair_similarities)

# 結果表示
for (normal, ono), sim in zip(pair_texts, pair_similarities):
    print(f"{sim:.3f}  {normal}  ⇔  {ono}")

print(f"\nペア数: {len(pair_similarities)}")
print(f"平均: {pair_similarities.mean():.4f}")
print(f"標準偏差: {pair_similarities.std():.4f}")
print(f"最小値: {pair_similarities.min():.4f}  (ペア: {pair_texts[np.argmin(pair_similarities)]})")
print(f"最大値: {pair_similarities.max():.4f}  (ペア: {pair_texts[np.argmax(pair_similarities)]})")

# 重複ペアのチェック(同一文字列が紛れていないか)
seen = {}
for i, (normal, ono) in enumerate(pair_texts):
    key = (normal, ono)
    if key in seen:
        print(f"警告: ペア {i} は ペア {seen[key]} と完全重複 → {key}")
    else:
        seen[key] = i


0.597  男の人が歩いている  ⇔  男の人がテクテク歩いている
0.718  帽子をかぶった男が歩いている  ⇔  帽子をかぶった男がスタスタ歩いている
0.937  杖をついた男が散歩している  ⇔  コツコツ杖をついた男が散歩している
1.000  杖をついた帽子の男が歩く  ⇔  杖をついた帽子の男が歩く
0.558  男が出かけている  ⇔  男がフラッと出かけている
0.731  男が杖をついている  ⇔  男がカツカツと杖をついている
0.681  男が外出している  ⇔  男がフラリと外出している
0.444  男がどこかに行くところ  ⇔  男がフラフラどこかに行くところ
0.365  男性が歩いている  ⇔  トコトコ歩く
0.185  帽子をかぶった男性が歩いている  ⇔  スタスタ歩く
0.540  杖を持った男性が歩いている  ⇔  男性がトコトコ歩いている
0.689  帽子をかぶって杖を持った男性が歩いている  ⇔  帽子をかぶった男性がトコトコ歩いている
0.448  歩いている人がいる  ⇔  帽子をかぶって杖を持った男性がトコトコ歩いている
0.521  今日は歩いていた  ⇔  今日はトコトコ歩こう
0.274  ある日、外に出かけた  ⇔  ステッキを持ってスタスタ歩こう
0.431  杖を持って外に出た  ⇔  外に出てスタスタ歩いている男性
0.379  外に出かけて歩く  ⇔  ある日、スタスタ歩いている男性がいた
0.432  今日は歩いてみよう  ⇔  タッタッと歩く男性
0.444  帽子をかぶった男性が歩いている  ⇔  トコトコ歩いている男の人
0.373  帽子をかぶった人が歩いている  ⇔  トコトコ歩いている男性
0.863  男性が杖をついて歩いている  ⇔  杖をついてトコトコ歩いている男性
0.511  男性がステッキをついて歩いている  ⇔  トットッと歩く男性
0.447  男が歩いている  ⇔  スタスタ歩く男性
0.591  男性が散歩している  ⇔  トコトコ軽快に歩く男性
0.493  散歩をしている男性がいる  ⇔  男の人はトコトコ歩いている
0.380  主人公は歩いている  ⇔  スタスタと杖を突いて歩く男性

ペア数: 26
平均: 0.5396
標準偏差: 0.1920


In [11]:
from sklearn.metrics.pairwise import cosine_similarity

sentence_embeddings = model.encode(sentences, batch_size=8)
print(cosine_similarity(sentence_embeddings))
#print(cosine_similarity([sentence_embeddings[0]], [sentence_embeddings[1]]))

[[1.       0.487518]
 [0.487518 1.      ]]


In [4]:
import sys
MrS_pairs = [('爪を深く切って欲しい', 'ぼごっとしてください'), ('マッサージをしている', 'もみもみしよらす'), ('いちゃいちゃしていたのではないのか', 'ちくちくしよったとだろ'), ('かっこいいですね', 'ばりっとしとる'), ('もやもやする？', 'もじゃもじゃする'), ('異性といちゃいちゃ(こそこそ)していたのではないか', 'こちょこちょしよったとでしょ'), ('シャキッとしている', 'ばりっとしとらす'), ('しっかり切ってください', 'ぴしゃっとしてください'), ('しっかり切ってくれるからよい', 'ぴしゃっとしてくれるけんよか'), ('どんどん（速度？）しているから大丈夫です', 'びゃんびゃんするけんよか'), ('眠りそうになっていた', 'うつらうつらしよった'), ('爪はしっかり切っておかないといけませんよ', 'せんせい！ばりっとしとかなんですよ'), ('自由に寝てもよいですよ？', 'ぶらっと寝なっせ'), ('シャッターを切ってください', 'ばちっと撮ってください'), ('カレンダーをめくらなければいけませんよ', 'ぱくっとせなんっですよ'), ('鉄砲で撃ちますよ', 'ぱんぱんぱんぱんてするですよ'), ('効果音？', 'ばぶっとせなん'), ('眠ってるんだろう', 'ぐーぐーしよらすたい'), ('かっこいい', 'ばりっとしとらす'), ('かっこいい（上をさらに強調して）', 'ばりばりしとらす'), ('言葉がすぐに出るならいいのに', 'ぱっとでるならよかばってん'), ('ちゃんと大便が出ないといけない', 'ぴしゃっと出らないかん'), ('大便が少しだけ出た', 'ちょこっと出た'), ('しっかり覚えている', 'びゃんびゃん覚えとる'), ('電話しようとするがボタンと押すのが間に合わず切れてしまう', 'ぷわーぷわーってなるとたい'), ('とてもたまる', 'びゃんびゃんたまる'), ('怒りたくなる？いらいら？', 'ぐらぐらしよごたる'), ('激しく文句を言いたい', 'ぎゃーぎゃー言おごたる'), ('大便がたくさん出た', 'がぼがぼ出た'), ('激しく文句を言いたい', 'ぎゃーぎゃー言おごたる'), ('時々は行くけれども', 'ちょこちょこは行くばってん'), ('びゃんびゃんするとは＝PT訓練', 'びゃんびゃんするとはなんだったかな'), ('あまりないから（ポツポツしかないから？）', 'ぼとぼとしかなかけん'), ('思い切り切ってよい', 'ばくっと切ってよか'), ('言葉が出たかと思うと続けて出てこない', 'ぱかって出るかと思うとばばっと出ない'), ('寝ている', 'ねんねしとらす'), ('きちんとしないといけない', 'ぴしゃってせなんとたい'), ('元気だけどとても元気というわけではない', 'げんきばってんびゃんびゃんはなか'), ('激しく嘔吐した', 'うぇーうぇー吐いた'), ('どんどんしないと', 'ぎゃんぎゃんせなん'), ('こういう風に・・・と使います', 'こがんしてがーがーぱっぱてします'), ('どんどん言えるもん', 'びゃんびゃん言いきるもん'), ('どんどんは言えないけれども', 'びゃんびゃんは言いきらんばってん'), ('すぱすぱ吸っていた？', 'ぱこぱこ吸いよった'), ('くしゃみが出る', 'へくしゃんの出る'), ('太ったんでしょ，いや，間違えた', 'ぽっちゃりでしょ，あら，しもた'), ('あら，ばきっといった', 'あら，ばぐっていうた'), ('少しだもんな', 'ぼそっとだもんな'), ('ぱっとでないんですよ', 'ばっとでらんとですよ'), ('？', 'ばりっとしとらんけん'), ('くしゃみが出なかった', 'はくしょんのでらんかった'), ('みそしるが??しているのがいいですね', 'みそしるのぱっとしとるのがよかですね'), ('昔は煙草をよく吸っていた', '昔はぱっかぱっか吸いよったたい'), ('きれいにしていないのはきらいだ', 'ぴしゃっとしとかんとすかん'), ('またボールを回す課題をしないといけない', 'またぱかぱかぱかぱかせなんですよ'), ('どんどん発言しないといけない', 'びゃんびゃんせなん'), ('どんどんしてもらったから', 'びゃんびゃんしてもろうたけん'), ('すぐにはわからないですよ，あれを見ても', 'ぱっとなわからんですよ，あれば見ても'), ('くしゃみをされましたよ', 'へくしゃんてしなはったばい'), ('涙がだらっと出る', '涙がだらっと出る'), ('たくさんぼろぼろと出る', 'たいぎゃなぼろぼろ出る'), ('とてもよい（調子）です', 'びゃんびゃんよかです'), ('げっそり痩せた', 'ごっそりなった'), ('（床が）つるつるしてるではないですか', 'つるつるしとるじゃなかですか'), ('つるつるしていたらいけませんよ', 'つるつるしとったらいかんですよ'), ('ふとした時が出ないんですよ', 'ぱっとしたときが出らんとですよ'), ('にこにこしておられる', 'にこにこしとらす'), ('倒れますかね', 'ばたってなりますかね'), ('ごわごわしている？', 'ぶわぶわしとる')]

In [ ]:
for pair in MrS_pairs:
    print(pair, end="")
    sentence_embeddings = model.encode(pair, batch_size=2)
    print(f'相関係数:{pearsonr(sentence_embeddings[0], sentence_embeddings[1])[0]:.3f}')



('爪を深く切って欲しい', 'ぼごっとしてください')相関係数:0.312
('マッサージをしている', 'もみもみしよらす')相関係数:0.242
('いちゃいちゃしていたのではないのか', 'ちくちくしよったとだろ')相関係数:0.518
('かっこいいですね', 'ばりっとしとる')相関係数:0.327
('もやもやする？', 'もじゃもじゃする')相関係数:0.466
('異性といちゃいちゃ(こそこそ)していたのではないか', 'こちょこちょしよったとでしょ')相関係数:0.448
('シャキッとしている', 'ばりっとしとらす')相関係数:0.549
('しっかり切ってください', 'ぴしゃっとしてください')相関係数:0.402
('しっかり切ってくれるからよい', 'ぴしゃっとしてくれるけんよか')相関係数:0.390
('どんどん（速度？）しているから大丈夫です', 'びゃんびゃんするけんよか')相関係数:0.269
('眠りそうになっていた', 'うつらうつらしよった')相関係数:0.431
('爪はしっかり切っておかないといけませんよ', 'せんせい！ばりっとしとかなんですよ')相関係数:0.258
('自由に寝てもよいですよ？', 'ぶらっと寝なっせ')相関係数:0.476
('シャッターを切ってください', 'ばちっと撮ってください')相関係数:0.313
('カレンダーをめくらなければいけませんよ', 'ぱくっとせなんっですよ')相関係数:0.107
('鉄砲で撃ちますよ', 'ぱんぱんぱんぱんてするですよ')相関係数:0.353
('効果音？', 'ばぶっとせなん')相関係数:0.423
('眠ってるんだろう', 'ぐーぐーしよらすたい')相関係数:0.188
('かっこいい', 'ばりっとしとらす')相関係数:0.356
('かっこいい（上をさらに強調して）', 'ばりばりしとらす')相関係数:0.469
('言葉がすぐに出るならいいのに', 'ぱっとでるならよかばってん')相関係数:0.472
('ちゃんと大便が出ないといけない', 'ぴしゃっと出らないかん')相関係数:0.523
('大便が少しだけ出た', 'ちょこっと出た')相関係数:0.480
('しっかり覚えている', 'びゃんびゃん覚えとる')相関係数:0.578
('電